# DFT (GPAW) untuk 5 Graphene-TPMS: Ground-State SCF, Band Gap, DOS, E_form, Adsorpsi Seluruh Polisulfida (S₈ → Li₂S₂), & Modulus Elastisitas (Tanpa Relaksasi)
Struktur TPMS: **Neovius, Primitive, IWP, Gyroid, Diamond** (murni C, P1).
Spesies Adsorbat Polisulfida Lengkap: **S₈, Li₂S₈, Li₂S₆, Li₂S₄, Li₂S₂** (5 spesies siklus reaksi baterai Li-S).

**Alur Komputasi Single-Point Ground-State (Direct SCF — Tanpa Relaksasi Geometri):**
1. **Langkah 1**: Perhitungan Ground-State SCF murni pada struktur pristine TPMS untuk mendapatkan energi total ({total}$), gaya residual ({max}$), dan tegangan kisi.
2. **Langkah 2**: Band gap dan DOS langsung dari wavefunction ground-state.
3. **Langkah 3**: Energi formasi ({form}$) relatif terhadap lembaran 2D graphene ({form} = E_{total}/N - E_{gr}$).
4. **Langkah 4**: Evaluasi afinitas penjeratan seluruh 5 spesies polisulfida ( 	o Li_2S_2$) di dalam pori TPMS ({ads} = E_{complex} - E_{host} - E_{gas}$).
5. **Langkah 5**: Modulus Bulk ($) dan Shear ($) dari tensor elastisitas / EOS.
6. **Langkah 6**: Ekspor ringkasan 5-target ke CSV dan visualisasi Profil Adsorpsi Multi-Spesies.

**Catatan**: Seluruh output, database JSON, log perhitungan, dan grafik otomatis disimpan ke direktori .


In [ ]:
import json, time, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

from ase.io import read, write
from ase.build import graphene
from ase.units import GPa
from ase.dft.bandgap import bandgap
from ase.dft.dos import DOS
from gpaw import GPAW, PW, FermiDirac

# ------------------------- DETEKSI DIREKTORI -------------------------
cif_cands = [
    Path("/media/user/uid1083/graphene_tpms"),
    Path("/home/user/Amarus/cgcnn_data_multiproperty/graphene_tpms"),
    Path("graphene_tpms"),
    Path(".")
]
CIF_DIR = next((p for p in cif_cands if (p / "graphene_sheet_neovius.cif").exists()), Path("."))

ads_cands = [
    Path("/media/user/uid1083/cifs_graphene_tpms_adsorbate"),
    Path("/home/user/Amarus/cgcnn_data_multiproperty/cifs_graphene_tpms_adsorbate"),
    Path("../cifs_graphene_tpms_adsorbate"),
    Path("cifs_graphene_tpms_adsorbate")
]
ADS_DIR = next((p for p in ads_cands if (p / "graphene_neovius_S8.cif").exists()), Path("."))

work_cands = [
    Path("/media/user/uid1083/dft_gpaw_graphene_tpms_testing/results"),
    Path("/home/user/Amarus/cgcnn_data_multiproperty/dft_gpaw_graphene_tpms_testing/results"),
    Path("results")
]
WORK = next((p for p in work_cands if p.parent.exists()), Path("results"))
WORK.mkdir(exist_ok=True, parents=True)
RES_FILE = WORK / "results.json"
RES = json.loads(RES_FILE.read_text()) if RES_FILE.exists() else {}
def save_res(): RES_FILE.write_text(json.dumps(RES, indent=2))
def rec(name): return RES.setdefault(name, {})

# ------------------------- PARAMETER FISIKA DFT -------------------------
POLYSULFIDES = ["S8", "Li2S8", "Li2S6", "Li2S4", "Li2S2"]
STRUCTS = {
    "neovius":   "graphene_sheet_neovius.cif",
    "primitive": "graphene_sheet_primitive.cif",
    "iwp":       "graphene_sheet_iwp.cif",
    "gyroid":    "graphene_sheet_gyroid.cif",
    "diamond":   "graphene_sheet_diamond.cif",
}
RUN = ["neovius", "primitive", "iwp", "gyroid", "diamond"]

# Mode: "lcao" (ultra-cepat, ~20-30 mnt) atau "pw" (Plane-Wave 400 eV)
DFT_MODE = "lcao"
ECUT, XC = 400, "PBE"
FORCE = False

def make_calc(atoms, txt, kpts=None, molecule=False, spin=False):
    if kpts is None:
        kpts = (1, 1, 1)
    if DFT_MODE == "lcao":
        return GPAW(mode="lcao", basis="dzp", xc=XC, kpts=kpts, spinpol=spin,
                    occupations=FermiDirac(0.01 if molecule else 0.05),
                    convergence={"energy": 1e-4, "density": 1e-3}, txt=str(txt))
    else:
        return GPAW(mode=PW(ECUT), xc=XC, kpts=kpts, spinpol=spin,
                    occupations=FermiDirac(0.01 if molecule else 0.05),
                    convergence={"energy": 1e-5, "density": 1e-4}, txt=str(txt))

def load_initial(name): return read(str(CIF_DIR / STRUCTS[name]))

print("✅ Konfigurasi Siap!")
print(f"   Mode DFT     : {DFT_MODE.upper()}")
print(f"   Daftar TPMS  : {RUN}")
print(f"   Output Folder: {WORK}")


## Langkah 1: Ground-State SCF Pristine (Single-Point — Tanpa Relaksasi)
* Menghitung SCF ground-state langsung pada koordinat pristine dari struktur TPMS.
* Menghitung energi total ({total}$), gaya residual atom ({max}$), dan stress kisi.
* Wavefunction ground-state disimpan ke .


In [ ]:
def scf_ground_state(name):
    r = rec(name)
    gpw = WORK / f"{name}.gpw"
    if gpw.exists() and not FORCE and "E_total" in r:
        print(f"✨ [{name}] SCF sudah ada (E = {r.get('E_total'):.4f} eV). Dilewati.")
        return
    a = load_initial(name)
    print("=" * 70)
    print(f"🚀 GROUND-STATE SCF: {name.upper()} ({len(a)} ATOM C) [MODE: {DFT_MODE.upper()}]")
    print("=" * 70)
    t0 = time.time()
    calc = make_calc(a, WORK / f"{name}_scf.txt")
    a.calc = calc
    E_tot = float(a.get_potential_energy())
    F = a.get_forces()
    fmax = float(np.linalg.norm(F, axis=1).max())
    try: stress_GPa = [float(x) for x in a.get_stress() / GPa]
    except: stress_GPa = [0.0] * 6
    r.update(n_atoms=len(a), cell_A=float(a.cell.lengths()[0]), E_total=E_tot, fmax=fmax,
             stress_GPa=stress_GPa, stress_max_GPa=float(np.abs(stress_GPa).max()), scf_time_s=float(time.time() - t0))
    a.calc.write(str(gpw), mode="all")
    save_res()
    print(f"✅ [{name}] Selesai ({time.time() - t0:.1f} s) | E = {E_tot:.4f} eV | fmax = {fmax:.3f} eV/Å")

for n in RUN: scf_ground_state(n)
df_step1 = pd.DataFrame(RES).T
cols_show = [c for c in ["n_atoms", "fmax", "E_total", "scf_time_s"] if c in df_step1.columns]
display(df_step1.loc[[n for n in RUN if n in df_step1.index], cols_show].round(3))


## Langkah 2: Band gap dan DOS
Perhitungan elektronik langsung dari ground-state wavefunction .

In [ ]:
def gap_dos(name):
    r = rec(name)
    gpw = WORK / f"{name}.gpw"
    if not gpw.exists():
        print(f"⚠️ [{name}] {gpw.name} belum ada, jalankan Langkah 1 dulu!")
        return
    if "gap_indirect" in r and not FORCE:
        print(f"✨ [{name}] Band gap & DOS sudah ada.")
        return
    calc = GPAW(str(gpw), txt=None)
    ef = calc.get_fermi_level()
    g, _, _ = bandgap(calc, direct=False, efermi=ef)
    gd, _, _ = bandgap(calc, direct=True, efermi=ef)
    try:
        dos = DOS(calc, width=0.1, npts=2000)
        e, d = dos.get_energies() - ef, dos.get_dos()
        np.savetxt(WORK / f"{name}_dos.dat", np.c_[e, d], header="E-EF(eV) DOS(states/eV)")
    except Exception as exc:
        print(f"   [Warn DOS {name}]: {exc}")
    r.update(E_fermi=float(ef), gap_indirect=float(g), gap_direct=float(gd))
    save_res()
    print(f"✅ [{name}] E_F={ef:.3f} eV | gap indirect={g:.3f} eV | direct={gd:.3f} eV")

# 1. Hitung bandgap & DOS untuk semua TPMS
for n in RUN:
    gap_dos(n)

# 2. Plot DOS & Band Gap terpisah PER-TPMS (Individual Plot)
for n in RUN:
    f = WORK / f"{n}_dos.dat"
    if not f.exists():
        continue
    e, d = np.loadtxt(f, unpack=True)
    n_atoms = RES.get(n, {}).get("n_atoms", 1)
    d_norm = d / n_atoms
    
    ef = RES.get(n, {}).get("E_fermi", 0.0)
    g_ind = RES.get(n, {}).get("gap_indirect", 0.0)
    g_dir = RES.get(n, {}).get("gap_direct", 0.0)
    
    fig, ax = plt.subplots(figsize=(7, 4.5))
    ax.plot(e, d_norm, color="#1d4ed8", lw=1.8, label=f"DOS ({n.capitalize()})")
    ax.fill_between(e, 0, d_norm, where=(e <= 0), color="#3b82f6", alpha=0.35, label="Valence Band (Occupied)")
    ax.fill_between(e, 0, d_norm, where=(e > 0), color="#94a3b8", alpha=0.15, label="Conduction Band (Empty)")
    ax.axvline(0, ls="--", color="#dc2626", lw=1.5, label="Fermi Level (E_F = 0 eV)")
    
    ax.set_xlim(-8, 8)
    ax.set_ylim(bottom=0)
    ax.set_xlabel("E - E_F (eV)", fontsize=11, fontweight="bold")
    ax.set_ylabel("DOS (states/eV/atom)", fontsize=11, fontweight="bold")
    ax.set_title(f"Electronic Density of States (DOS) - Graphene {n.capitalize()}", fontsize=12, fontweight="bold")
    
    info = (
        f"Topology: {n.upper()}\n"
        f"Fermi Level: {ef:.3f} eV\n"
        f"Indirect Gap: {g_ind:.3f} eV\n"
        f"Direct Gap: {g_dir:.3f} eV\n"
        f"Electronic Type: {Semimetal if g_ind < 0.1 else Semiconductor}"
    )
    ax.text(0.96, 0.94, info, transform=ax.transAxes, fontsize=9.5,
            verticalalignment="top", horizontalalignment="right",
            bbox=dict(boxstyle="round,pad=0.5", facecolor="#f8fafc", edgecolor="#cbd5e1", alpha=0.9))
    
    ax.legend(loc="upper left", frameon=True, fontsize=9)
    ax.grid(True, ls=":", alpha=0.6)
    plt.tight_layout()
    out_img = WORK / f"dos_{n}.png"
    plt.savefig(out_img, dpi=200)
    plt.show()


## Langkah 3: Energi formasi relatif terhadap 2D Graphene (eV/atom)
Material murni C → referensi 2D graphene lembar datar ({form} = E_{total}/N - E_{gr}$).


In [ ]:
def graphene_ref():
    if "graphene_ref" in RES and not FORCE: return RES["graphene_ref"]["E_per_atom"]
    g = graphene(formula="C2", a=2.46, vacuum=8.0); g.pbc = True
    g.calc = make_calc(g, WORK / "graphene_ref.txt", kpts=(6, 6, 1))
    E = g.get_potential_energy() / len(g)
    RES["graphene_ref"] = {"E_per_atom": E}; save_res(); return E

E_gr = graphene_ref(); print(f"E(graphene) = {E_gr:.4f} eV/atom")
for n in RUN:
    if "E_total" in RES.get(n, {}):
        rec(n)["E_form_eV_atom"] = RES[n]["E_total"] / RES[n]["n_atoms"] - E_gr
        print(f"{n}: E_form = {RES[n]['E_form_eV_atom']:.4f} eV/atom")
save_res()


## Langkah 4: Evaluasi Adsorpsi Seluruh Produk Polisulfida (S₈, Li₂S₈, Li₂S₆, Li₂S₄, Li₂S₂)
Menghitung energi penjeratan termodinamika secara konsisten (Single-Point SCF):
1704356E_{ads}(mol) = E(TPMS + mol) - E(TPMS) - E_{gas}(mol)1704356


In [ ]:
def free_molecule(mol_name):
    tag = f"gas_{mol_name}"
    xyz_path = WORK / f"{tag}.xyz"
    if tag in RES and xyz_path.exists() and not FORCE:
        return read(str(xyz_path)), RES[tag]["E_free"]
    ref_cif = ADS_DIR / f"graphene_neovius_{mol_name}.cif"
    cplx = read(str(ref_cif))
    symbols = cplx.get_chemical_symbols()
    mol_atoms = cplx[[i for i, s in enumerate(symbols) if s != "C"]]
    mol_atoms.center(vacuum=10.0)
    mol_atoms.pbc = True
    print(f"   [Gas DFT] SCF molekul fasa gas: {mol_name} ({mol_atoms.get_chemical_formula()})...")
    mol_atoms.calc = make_calc(mol_atoms, WORK / f"{tag}_scf.txt", molecule=True)
    E_free = float(mol_atoms.get_potential_energy())
    mol_atoms.calc = None
    write(str(xyz_path), mol_atoms)
    RES[tag] = {"E_free": E_free, "formula": mol_atoms.get_chemical_formula()}
    save_res()
    print(f"   ✅ {mol_name} gas: E = {E_free:.4f} eV")
    return mol_atoms, E_free

def adsorb_all_polysulfides(name):
    r = rec(name)
    E_host = r.get("E_total")
    if E_host is None:
        print(f"⚠️ [{name}] E_total host belum ada. Jalankan Langkah 1 terlebih dahulu!")
        return
    print("=" * 70)
    print(f"🚀 EVALUASI ADSORPSI 5 POLISULFIDA: {name.upper()} [SINGLE-POINT SCF]")
    print("=" * 70)
    out_eads = {}
    for mol in POLYSULFIDES:
        key = f"Eads_{mol}"
        cif_out = WORK / f"{name}_{mol}_adsorbed.cif"
        if key in r and cif_out.exists() and not FORCE:
            print(f"   ✨ [{name} + {mol}] Sudah ada -> E_ads = {r[key]:.3f} eV")
            out_eads[mol] = r[key]
            continue
        _, E_mol_gas = free_molecule(mol)
        cplx_cif = ADS_DIR / f"graphene_{name}_{mol}.cif"
        if not cplx_cif.exists():
            print(f"   ⚠️ File {cplx_cif.name} tidak ditemukan, dilewati.")
            continue
        cplx = read(str(cplx_cif))
        t0 = time.time()
        cplx.calc = make_calc(cplx, WORK / f"{name}_{mol}_scf.txt")
        E_cplx = float(cplx.get_potential_energy())
        E_ads = float(E_cplx - E_host - E_mol_gas)
        r[key] = E_ads
        out_eads[mol] = E_ads
        write(str(cif_out), cplx)
        save_res()
        print(f"   ✅ {name} + {mol}: E_ads = {E_ads:.3f} eV ({time.time() - t0:.1f} s)")
    if out_eads:
        r["Eads_mean"] = float(np.mean(list(out_eads.values())))
        r["Eads_best"] = float(min(out_eads.values()))
        save_res()

for nm in RUN: adsorb_all_polysulfides(nm)
df_ads = pd.DataFrame(RES).T
ads_cols = [f"Eads_{m}" for m in POLYSULFIDES if f"Eads_{m}" in df_ads.columns] + ["Eads_mean", "Eads_best"]
ads_cols_exist = [c for c in ads_cols if c in df_ads.columns]
if ads_cols_exist:
    print("
📋 TABEL ENERGI ADSORPSI POLISULFIDA (eV):")
    display(df_ads.loc[[n for n in RUN if n in df_ads.index], ads_cols_exist].round(3))


## Langkah 5: Modulus Bulk (K) dan Shear (G) via Regangan Kisi (Clamped-Ion)
* Mode LCAO: Bulk modulus dihitung via Birch-Murnaghan EOS ($ vs volume), Shear modulus dihitung via pure shear strain energy.
* Mode PW: Dihitung via tensor elastisitas {ij}$ 6×6.


In [ ]:
def elastic(name):
    r = rec(name)
    if "K_VRH" in r and not FORCE: print(f"✨ [{name}] Modulus sudah ada."); return
    base = load_initial(name); v0 = float(base.get_volume()); e0 = float(r.get("E_total", 0.0))
    if DFT_MODE == "lcao":
        print(f"🔩 [{name}] Menghitung K (Birch-Murnaghan EOS) & G (Pure Shear Strain)...
")
        scale_factors = [0.98**(1/3), 0.99**(1/3), 1.00, 1.01**(1/3), 1.02**(1/3)]
        vols, energies = [], []
        for sf in scale_factors:
            v_s = float(v0 * (sf**3)); vols.append(v_s)
            if abs(sf - 1.0) < 1e-4: energies.append(e0)
            else:
                at_s = base.copy(); at_s.set_cell(at_s.cell * sf, scale_atoms=True)
                at_s.calc = make_calc(at_s, WORK / f"{name}_eos_sf{sf**3:.3f}.txt")
                energies.append(float(at_s.get_potential_energy()))
        coeffs = np.polyfit(vols, energies, 2)
        K_gpa = float(v0 * (2.0 * coeffs[0]) * 160.21766)
        gammas = [-0.015, -0.008, 0.0, 0.008, 0.015]; e_shears = []
        for g in gammas:
            if abs(g) < 1e-5: e_shears.append(0.0)
            else:
                at_g = base.copy(); cell_o = base.cell.copy()
                sm = np.array([[1.0, g, 0.0], [g, 1.0, 0.0], [0.0, 0.0, 1.0 / (1.0 - g**2)]])
                at_g.set_cell(cell_o @ sm.T, scale_atoms=True)
                at_g.calc = make_calc(at_g, WORK / f"{name}_shear_g{g:+.3f}.txt")
                e_shears.append(float(at_g.get_potential_energy()) - e0)
        poly_g = np.polyfit(np.array(gammas)**2, e_shears, 1)
        G_gpa = float(poly_g[0] / (0.5 * v0) * 160.21766)
        E_young = float(9*K_gpa*G_gpa / (3*K_gpa + G_gpa)) if (3*K_gpa + G_gpa) > 0 else 0.0
        poisson = float((3*K_gpa - 2*G_gpa) / (2*(3*K_gpa + G_gpa))) if (3*K_gpa + G_gpa) > 0 else 0.0
        r.update(K_VRH=round(K_gpa, 2), G_VRH=round(G_gpa, 2), E_young=round(E_young, 2), poisson=round(poisson, 3))
        save_res(); print(f"✅ [{name}] K = {K_gpa:.1f} GPa | G = {G_gpa:.1f} GPa")
        return
for n in RUN: elastic(n)


## Langkah 6: Ringkasan 5 Sifat Fisik & Visualisasi Profil Adsorpsi Lengkap
Menyimpan seluruh hasil komputasi ke  dan membuat visualisasi grafik.


In [ ]:
df = pd.DataFrame({k: v for k, v in RES.items() if k in STRUCTS}).T
cols_phys = [c for c in ["n_atoms","fmax","gap_indirect","gap_direct","E_form_eV_atom","K_VRH","G_VRH","E_young","poisson"] if c in df.columns]
cols_ads = [c for c in [f"Eads_{m}" for m in POLYSULFIDES] + ["Eads_mean", "Eads_best"] if c in df.columns]
summary = df[cols_phys + cols_ads].astype(float).round(3)
summary_csv = WORK / "summary.csv"
summary.to_csv(summary_csv)
print(f"✅ Ringkasan CSV tersimpan ke: {summary_csv}")
display(summary)

# Visualisasi Profil Adsorpsi Lengkap
ads_species_cols = [f"Eads_{m}" for m in POLYSULFIDES if f"Eads_{m}" in summary.columns]
if ads_species_cols:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    species_labels = [c.replace("Eads_", "") for c in ads_species_cols]
    for idx, row in summary.iterrows():
        vals = [row[c] for c in ads_species_cols]
        ax.plot(species_labels, vals, marker="o", linewidth=2, label=str(idx).upper())
    ax.axhline(0, color="gray", linestyle="--", alpha=0.7)
    ax.set_xlabel("Spesies Polisulfida (Siklus Litiasi Redoks)", fontweight="bold")
    ax.set_ylabel("Energi Adsorpsi {ads}$ (eV)", fontweight="bold")
    ax.set_title("Profil Afinitas Penjeratan Polisulfida pada Graphene TPMS", fontweight="bold", fontsize=12)
    ax.grid(True, linestyle=":", alpha=0.6); ax.legend(); plt.tight_layout()
    prof_png = WORK / "adsorption_profile_5species.png"
    plt.savefig(prof_png, dpi=200); plt.show()
    print(f"📊 Grafik Profil Adsorpsi tersimpan ke: {prof_png}")
